In [1]:
# This cell is removed with the tag: "remove-input"
# As such, it will not be shown in documentation

import warnings
warnings.filterwarnings('ignore')

(Tutorial_Add)=
# Add

*Adding elements of a molecular system into another molecular system.*

Elements coming from different molecular systems can be added to a given system with the {func}`molsysmt.basic.add` function.

:::{versionadded} 1.0.0
:::

:::{admonition} API documentation
:class: dropdown

Follow this link for a detailed description of the input arguments, raised errors, and returned objects of this function: {func}`molsysmt.basic.add`.
:::


## Basic usage

Let's show how this function works with three peptides defined as three different molecular systems: proline dipeptide ($A$), valine dipeptide ($B$), and lysine dipeptide ($C$).


In [2]:
import molsysmt as msm

In [3]:
molsys_A = msm.build.build_peptide('AceProNme')
molsys_B = msm.build.build_peptide('AceValNme')
molsys_C = msm.build.build_peptide('AceLysNme')

In [4]:
molsys_B = msm.structure.translate(molsys_B, translation='[-1.0, 0.0, 0.0] nanometers')
molsys_C = msm.structure.translate(molsys_C, translation='[1.0, 0.0, 0.0] nanometers')

Let's inspect how $A$ is defined before adding $B$ and $C$:

In [5]:
msm.info(molsys_A)

form,n_atoms,n_groups,n_components,n_chains,n_molecules,n_entities,n_peptides,n_structures
molsysmt.MolSys,26,3,1,1,1,1,1,1


Now let's add the elements of $B$ and $C$ into $A$:

In [6]:
msm.add(molsys_A, molsys_B)
msm.add(molsys_A, molsys_C)

:::{tip}
:class: dropdown

All methods defined in the {ref}`molsysmt.basic <API basic>` module can be invoked also from the main level of the library. Hence, {func}`molsysmt.add` is the same method as {func}`molsysmt.basic.add`.
:::

After the addition, we inspect $A$ again. Notice the larger count of atoms, groups, and molecules:

In [7]:
msm.info(molsys_A, element='system')

form,n_atoms,n_groups,n_components,n_chains,n_molecules,n_entities,n_peptides,n_structures
molsysmt.MolSys,88,9,3,3,3,3,3,1


We can also visualize the combined system interactively. Try rotating and zooming to observe the spatial separation between $A$, $B$, and $C$.

In [8]:
# This cell is removed with the tag: "remove-input"
# As such, it will not be shown in documentation

molsysviewer_htmlfile = '_static/views/tools_basic_add.html'

In [9]:
msm.view(molsys_A)

'<iframe src="../../../../_static/views/tools_basic_add.html" width="100%" height="480px"\n        style="border:none;"></iframe>'

By default, `msm.add` modifies the target system in place. If you prefer not to modify the original system, pass `in_place=False` to return a new molecular system containing the combined elements:

In [10]:
molsys_D = msm.add(molsys_B, molsys_C, in_place=False)

In [11]:
msm.get(molsys_B, n_peptides=True)

1

In [12]:
msm.get(molsys_C, n_peptides=True)

1

In [13]:
msm.get(molsys_D, n_peptides=True)

2

## Adding selected elements

Instead of adding all elements from the source system, you can restrict the addition to a specific selection of atoms or residues using `selection`. For example, let's add only the Lysine residue from $C$ into $B$:

In [14]:
molsys_E = msm.add(molsys_B, molsys_C, selection='group_name=="LYS"', in_place=False)

In [15]:
msm.info(molsys_E)

form,n_atoms,n_groups,n_components,n_chains,n_molecules,n_entities,n_peptides,n_structures
molsysmt.MolSys,50,4,2,2,2,2,2,1


## Specifying structure indices

When adding elements from a multi-structure source system into a target system with fewer structures, pass `structure_indices` to select which frame's coordinates to add:

In [16]:
molsys_A1 = msm.build.build_peptide('AceProNme')
molsys_A2 = msm.structure.translate(molsys_A1, translation='[0.1, 0.1, 0.1] nanometers')
molsys_multi = msm.concatenate_structures([molsys_A1, molsys_A2])
molsys_F = msm.add(molsys_B, molsys_multi, structure_indices=0, in_place=False)

In [17]:
msm.info(molsys_F)

form,n_atoms,n_groups,n_components,n_chains,n_molecules,n_entities,n_peptides,n_structures
molsysmt.MolSys,54,6,2,2,2,2,2,1


## What is kept and what is dropped

Adding atoms changes the system, and not every piece of structural data can survive that change. MolSysMT decides by what each value describes, not by how it is stored.

**Data attached to each atom** — coordinates, velocities, B factors, occupancy, force-field parameters — is concatenated when both systems have it. When only one of the two has it, there is no honest way to build a column covering all the atoms of the result, so the whole attribute is dropped and a `StructuralAttributeDropWarning` says which ones.

**Data describing the structure axis** — `structure_id`, `time`, `time_step` — is untouched, because `msm.add` grows the atom axis and leaves the structure axis exactly as it was.

**The periodic box** stays the one of the target system: `msm.add` never reinterprets the unit cell. If the two systems disagree, or if one is periodic and the other is not, an `IncompatibleBoxWarning` reports it. Coordinates expressed under a different box are not directly comparable, and combining them quietly would hide that from you.

**Values describing the whole system** — `temperature`, `potential_energy`, `kinetic_energy` — are dropped. The energy of the target was computed for the target, and after adding atoms it is no longer the energy of anything. It is not the sum of the two either.

Let's see it with a real case: T4 lysozyme read from a PDB file, which carries B factors and a unit cell, and a small peptide built from scratch, which carries neither.

In [18]:
molsys_G = msm.convert(msm.systems['T4 lysozyme L99A']['181l.pdb'], to_form='molsysmt.MolSys')
molsys_H = msm.build.build_peptide('AceAlaNme')

print('lysozyme has B factors:', msm.get(molsys_G, b_factor=True) is not None)
print('built peptide has B factors:', msm.get(molsys_H, b_factor=True) is not None)

lysozyme has B factors: True
built peptide has B factors: False


:::{note} Demo Systems Catalog
:class: dropdown

This tutorial uses demonstration datasets provided by MolSysMT. To explore the full catalog of bundled systems, forms, and file paths, visit the {ref}`Demo Systems <user-foundations-entrance-demo-systems>` guide.
:::

In [19]:
molsys_I = msm.add(molsys_G, molsys_H, in_place=False)

print('atoms in the result:', msm.get(molsys_I, n_atoms=True))
print('B factors survived:', msm.get(molsys_I, b_factor=True) is not None)

atoms in the result: 1463
B factors survived: False


The atoms were added and the B factors are gone: the peptide had none, and a column describing only the lysozyme half of the result would not be a B-factor series.

### Refusing instead of dropping

If losing the target's data silently is not acceptable for your workflow, pass `attribute_policy='strict'`. The operation is then rejected before anything is modified, and both systems are left exactly as they were:

In [20]:
from molsysmt import StructuralInconsistencyError

try:
    msm.add(molsys_G, molsys_H, in_place=False, attribute_policy='strict')
except StructuralInconsistencyError as error:
    print('rejected:', error)

print('lysozyme still intact:', msm.get(molsys_G, n_atoms=True), 'atoms')

rejected: Structural inconsistency detected: These atom-aligned attributes are present on only one side, so the result would cover part of the atom axis: b_factor, occupancy; use attribute_policy='intersection' to discard them instead. Ensure that the atoms, residues, or frames match between the systems being compared or merged. Docs: https://www.uibcdf.org/MolSysMT
lysozyme still intact: 1441 atoms


:::{warning}
`msm.add` takes **one** target and **one** source. A list is read as a single molecular system split into complementary items — a topology file next to a coordinate file, exactly as {func}`molsysmt.basic.convert` reads it — and is assembled before the addition. It is never a sequence of systems to add one after another. To add several sources, call {func}`molsysmt.basic.add` once per source.

A molecular system given as complementary items cannot be grown in place, because the assembled result is a new object. Use `in_place=False` in that case.
:::

To combine several systems into a new one in a single call, see {func}`molsysmt.basic.merge` instead.

:::{seealso} Related Tools & References
:class: dropdown

- {ref}`Tutorial_Build_peptide`: Build natural peptides with or without terminal caps with {func}`molsysmt.build.build_peptide`.
- {ref}`Tutorial_Translate`: Translate molecular systems in space with {func}`molsysmt.structure.translate`.
- {ref}`Tutorial_Info`: Print a summary of the contents, topology, and structural data of a molecular system with {func}`molsysmt.basic.info`.
- {ref}`Tutorial_View`: Show a molecular system interactively in 3D with {func}`molsysmt.basic.view`.
- {ref}`Tutorial_Get`: Retrieve attribute values from a molecular system with {func}`molsysmt.basic.get`.
- {ref}`Tutorial_Select`: Select elements of a molecular system with {func}`molsysmt.basic.select`.
- {ref}`Tutorial_Concatenate_structures`: Concatenate the structures found in a list of molecular systems with {func}`molsysmt.basic.concatenate_structures`.
- {ref}`Tutorial_Convert`: Convert a molecular system into other form or forms with {func}`molsysmt.basic.convert`.
- {ref}`Tutorial_Merge`: Merge the elements of different molecular systems with {func}`molsysmt.basic.merge`.
:::